In [1]:
import os

# ==========================================
# CAPA DE DATOS (Data Layer)
# ==========================================
class DataLayer:
    def __init__(self, fileName):
        self.fileName = fileName

    def saveRecord(self, recordString):
        with open(self.fileName, "a", encoding="utf-8") as file:
            file.write(recordString + "\n")

    def readAllRecords(self):
        if not os.path.exists(self.fileName):
            return []
        with open(self.fileName, "r", encoding="utf-8") as file:
            return file.readlines()

# ==========================================
# CAPA DE NEGOCIO (Business Logic Layer)
# ==========================================
class BusinessLayer:
    def __init__(self):
        self.dataLayer = DataLayer("inspecciones.txt")

    def addInspection(self, inspectionId, conglomerateData, brigadeData, fustalesCount, latizalesCount, brinzalesCount, samplesList):
        # Se serializan las muestras recolectadas
        samplesString = "-".join([f"{s['idCode']}:{s['commonName']}:{s['useType']}:{s['scientificName']}" for s in samplesList])

        # Formato de texto plano separado por pipes (|)
        recordString = f"{inspectionId}|{conglomerateData}|{brigadeData}|{fustalesCount}|{latizalesCount}|{brinzalesCount}|{samplesString}"

        self.dataLayer.saveRecord(recordString)
        return True

    def searchInspection(self, searchId):
        allRecords = self.dataLayer.readAllRecords()
        for line in allRecords:
            dataFields = line.strip().split('|')
            if len(dataFields) >= 7 and dataFields[0] == searchId:
                return {
                    "inspectionId": dataFields[0],
                    "conglomerateData": dataFields[1],
                    "brigadeData": dataFields[2],
                    "fustalesCount": dataFields[3],
                    "latizalesCount": dataFields[4],
                    "brinzalesCount": dataFields[5],
                    "samplesList": dataFields[6]
                }
        return None

# ==========================================
# CAPA DE PRESENTACIÓN (Presentation Layer)
# ==========================================
class PresentationLayer:
    def __init__(self):
        self.businessLayer = BusinessLayer()

    def displayMainMenu(self):
        while True:
            print("\n--- INVENTARIO FORESTAL NACIONAL (IDEAM) ---")
            print("1. Registrar nueva inspección")
            print("2. Consultar inspección")
            print("3. Salir")

            userChoice = input("Seleccione una opción: ")

            if userChoice == "1":
                self.registerNewInspection()
            elif userChoice == "2":
                self.consultExistingInspection()
            elif userChoice == "3":
                print("Saliendo del sistema...")
                break
            else:
                print("Opción inválida. Intente de nuevo.")

    def registerNewInspection(self):
        print("\n--- REGISTRO DE INSPECCIÓN ---")
        inspectionId = input("ID de la inspección: ")
        conglomerateData = input("Datos del conglomerado (Ubicación/Centroide): ")
        brigadeData = input("Datos de la brigada encargada: ")

        try:
            fustalesCount = int(input("Cantidad de árboles Fustales: "))
            latizalesCount = int(input("Cantidad de árboles Latizales: "))
            brinzalesCount = int(input("Cantidad de árboles Brinzales: "))
        except ValueError:
            print("Error: El conteo de árboles debe ser numérico.")
            return

        samplesList = []
        addMoreSamples = "s"

        while addMoreSamples.lower() == "s":
            print("\n- Registro de Muestra -")
            idCode = input("Código de identificación de la muestra: ")
            commonName = input("Nombre común de la especie: ")
            useType = input("Uso (medicinal / alimenticio / maderable): ")
            scientificName = input("Nombre científico (o 'pendiente' si va a laboratorio): ")

            samplesList.append({
                "idCode": idCode,
                "commonName": commonName,
                "useType": useType,
                "scientificName": scientificName
            })
            addMoreSamples = input("¿Desea agregar otra muestra? (s/n): ")

        self.businessLayer.addInspection(
            inspectionId, conglomerateData, brigadeData,
            fustalesCount, latizalesCount, brinzalesCount, samplesList
        )
        print("¡Inspección guardada exitosamente en el archivo de texto!")

    def consultExistingInspection(self):
        print("\n--- CONSULTA DE INSPECCIÓN ---")
        searchId = input("Ingrese el ID de la inspección a consultar: ")
        inspectionResult = self.businessLayer.searchInspection(searchId)

        if inspectionResult:
            print("\n--- DATOS DE LA INSPECCIÓN ---")
            print(f"ID Inspección: {inspectionResult['inspectionId']}")
            print(f"Conglomerado: {inspectionResult['conglomerateData']}")
            print(f"Brigada: {inspectionResult['brigadeData']}")
            print(f"Conteo de árboles -> Fustales: {inspectionResult['fustalesCount']}, Latizales: {inspectionResult['latizalesCount']}, Brinzales: {inspectionResult['brinzalesCount']}")

            print("\nMuestras recolectadas:")
            if inspectionResult['samplesList']:
                samplesArray = inspectionResult['samplesList'].split('-')
                for sampleString in samplesArray:
                    sampleParts = sampleString.split(':')
                    if len(sampleParts) == 4:
                        print(f"  - Código: {sampleParts[0]} | Nombre Común: {sampleParts[1]} | Uso: {sampleParts[2]} | Científico: {sampleParts[3]}")
            else:
                print("  No se registraron muestras.")
        else:
            print("No se encontró ninguna inspección registrada con ese ID.")

# ==========================================
# EJECUCIÓN PRINCIPAL
# ==========================================
if __name__ == "__main__":
    appInstance = PresentationLayer()
    appInstance.displayMainMenu()


--- INVENTARIO FORESTAL NACIONAL (IDEAM) ---
1. Registrar nueva inspección
2. Consultar inspección
3. Salir


KeyboardInterrupt: Interrupted by user